# 08 — Transpiler Pipeline Validation

This notebook validates the `qc-compiler` QCompiler pipeline (transpiler.py) which composes all 6 optimization modules — autotuning, gate fusion, circuit cutting, coherence-aware scheduling, adaptive error mitigation, and circuit batching — into a single compilation pass.

## 1. Setup & Imports

In [ ]:
import qc_compiler
from qc_compiler import (
    QCompiler, OptimizerConfig, QCompilerResult,
    CostModel, GateFusion, CircuitCutter,
    AdaptiveErrorMitigation, CoherenceAwareScheduler,
    CircuitBatcher, AutoTuner,
)
from qiskit import QuantumCircuit
from qiskit.circuit.library import QFT
from qiskit_ibm_runtime.fake_provider import FakeBrisbane

print(f"qc-compiler version: {qc_compiler.__version__}")
print("All imports successful!")

## 2. Default Optimization Pipeline (All Passes Enabled)

In [ ]:
compiler = QCompiler()

bell = QuantumCircuit(2)
bell.h(0)
bell.cx(0, 1)
bell.measure_all()

result = compiler.optimize(bell)

print(f"Original: {result.original_circuit.num_qubits} qubits, depth={result.original_circuit.depth()}")
print(f"Optimized: {result.optimized_circuit.num_qubits} qubits, depth={result.optimized_circuit.depth()}")
print(f"Fidelity: {result.fidelity_before:.6f} -> {result.fidelity_after:.6f}")
print(f"Improvement: {result.fidelity_improvement:.6f} ({result.fidelity_improvement_pct:.2f}%)")
print(f"Passes applied: {result.passes_applied}")

assert isinstance(result, QCompilerResult)
assert result.original_circuit is not None
assert result.optimized_circuit is not None
assert result.fidelity_before > 0
assert result.fidelity_after > 0
assert len(result.passes_applied) > 0
print("\nAll default pipeline assertions passed!")

In [ ]:
ghz = QuantumCircuit(4)
ghz.h(0)
for i in range(1, 4):
    ghz.cx(0, i)
ghz.measure_all()

result_ghz = compiler.optimize(ghz)
print(f"GHZ-4 default pipeline:")
print(f"  Fidelity: {result_ghz.fidelity_before:.6f} -> {result_ghz.fidelity_after:.6f}")
print(f"  Improvement: {result_ghz.fidelity_improvement:.6f}")
print(f"  Passes: {result_ghz.passes_applied}")
print(f"  Config: fusion={result_ghz.config.fusion}, cutting={result_ghz.config.cutting}, "
      f"mitigation={result_ghz.config.mitigation}, scheduling={result_ghz.config.scheduling}")

assert result_ghz.fidelity_before > 0
assert result_ghz.fidelity_after > 0
assert result_ghz.config.fusion is True
assert result_ghz.config.cutting is True
assert result_ghz.config.mitigation == "adaptive"
assert result_ghz.config.scheduling == "coherence_aware"

## 3. Fusion-Only Optimization

In [ ]:
fusion_config = OptimizerConfig(
    fusion=True,
    cutting=False,
    mitigation="none",
    scheduling="none",
    batch=False,
    autotune=False,
)

qc_fusible = QuantumCircuit(2)
qc_fusible.h(0)
qc_fusible.rz(0.5, 0)
qc_fusible.sx(0)
qc_fusible.cx(0, 1)

result_fusion = compiler.optimize(qc_fusible, config=fusion_config)

print(f"Fusion-only pipeline:")
print(f"  Passes: {result_fusion.passes_applied}")
print(f"  Fidelity: {result_fusion.fidelity_before:.6f} -> {result_fusion.fidelity_after:.6f}")

if result_fusion.fusion_result is not None:
    fr = result_fusion.fusion_result
    print(f"  Chains fused: {fr.chains_fused}")
    print(f"  Gates: {fr.total_gates_before} -> {fr.total_gates_after}")
    print(f"  Depth: {fr.depth_before} -> {fr.depth_after}")

assert "fusion" in result_fusion.passes_applied
assert result_fusion.fusion_result is not None
assert result_fusion.cutting_result is None
assert result_fusion.mitigation_plan is None
assert result_fusion.schedule_result is None
print("\nFusion-only assertions passed!")

## 4. Scheduling-Only Optimization

In [ ]:
scheduling_methods = ["asap", "alap", "coherence_aware"]
qc_sched = QuantumCircuit(3)
qc_sched.h(0)
qc_sched.cx(0, 1)
qc_sched.cx(1, 2)

print(f"{'Method':<20} {'Depth':>5} {'Fidelity Before':>15} {'Fidelity After':>15}")
print("-" * 58)

for method in scheduling_methods:
    config = OptimizerConfig(
        fusion=False,
        cutting=False,
        mitigation="none",
        scheduling=method,
        batch=False,
        autotune=False,
    )
    result = compiler.optimize(qc_sched, config=config)
    print(f"{method:<20} {result.optimized_circuit.depth():>5} "
          f"{result.fidelity_before:>15.6f} {result.fidelity_after:>15.6f}")

    assert f"scheduling:{method}" in result.passes_applied
    assert result.schedule_result is not None
    assert result.fusion_result is None
    assert result.mitigation_plan is None

print("\nAll scheduling method assertions passed!")

In [ ]:
result_coherence = compiler.optimize(qc_sched, config=OptimizerConfig(
    fusion=False, cutting=False, mitigation="none",
    scheduling="coherence_aware", batch=False, autotune=False,
))

sr = result_coherence.schedule_result
print(f"Coherence-aware schedule detail:")
print(f"  Method: {sr.method}")
print(f"  Depth ASAP: {sr.depth_asap}, ALAP: {sr.depth_alap}, Optimized: {sr.depth_optimized}")
print(f"  Fidelity ASAP: {sr.estimated_fidelity_asap:.6f}")
print(f"  Fidelity ALAP: {sr.estimated_fidelity_alap:.6f}")
print(f"  Fidelity Opt: {sr.estimated_fidelity_optimized:.6f}")
print(f"  Idle time total: {sr.idle_time_total:.6e}s")
print(f"  Idle time avg: {sr.idle_time_avg:.6e}s")

assert sr.method == "coherence_aware"
assert sr.estimated_fidelity_optimized >= 0

## 5. Mitigation-Only Optimization

In [ ]:
mitigation_methods = ["zne", "pec", "cdr"]
qc_mitig = QuantumCircuit(2)
qc_mitig.h(0)
qc_mitig.cx(0, 1)

print(f"{'Method':<12} {'Total Shots':>11} {'Scales':>10} {'Segments':>8}")
print("-" * 45)

for method in mitigation_methods:
    config = OptimizerConfig(
        fusion=False,
        cutting=False,
        mitigation=method,
        scheduling="none",
        batch=False,
        autotune=False,
    )
    result = compiler.optimize(qc_mitig, config=config)
    plan = result.mitigation_plan

    print(f"{method:<12} {plan.total_shots:>11} {str(plan.noise_scales):>10} {plan.segments:>8}")

    assert f"mitigation:{method}" in result.passes_applied
    assert result.mitigation_plan is not None
    assert result.mitigation_plan.method == method
    assert result.fusion_result is None
    assert result.schedule_result is None

print("\nAll mitigation method assertions passed!")

In [ ]:
result_adaptive = compiler.optimize(qc_mitig, config=OptimizerConfig(
    fusion=False, cutting=False, mitigation="adaptive",
    scheduling="none", batch=False, autotune=False,
))

plan_adaptive = result_adaptive.mitigation_plan
print(f"Adaptive mitigation (resolves to zne internally):")
print(f"  Pass applied: {result_adaptive.passes_applied}")
print(f"  Plan method: {plan_adaptive.method}")
print(f"  Noise scales: {plan_adaptive.noise_scales}")
print(f"  Total shots: {plan_adaptive.total_shots}")

assert "mitigation:adaptive" in result_adaptive.passes_applied
assert plan_adaptive.method == "zne"
print("\nAdaptive mitigation assertion passed!")

## 6. Cutting-Only Optimization

In [ ]:
cutting_config = OptimizerConfig(
    fusion=False,
    cutting=True,
    mitigation="none",
    scheduling="none",
    batch=False,
    autotune=False,
)

qc_cut = QuantumCircuit(2)
qc_cut.h(0)
qc_cut.cx(0, 1)

result_cutting = compiler.optimize(qc_cut, config=cutting_config)

print(f"Cutting-only pipeline:")
print(f"  Passes: {result_cutting.passes_applied}")
print(f"  Fidelity: {result_cutting.fidelity_before:.6f} -> {result_cutting.fidelity_after:.6f}")

cr = result_cutting.cutting_result
if cr is not None:
    print(f"  Should cut: {cr.should_cut}")
    print(f"  Num cuts: {cr.num_cuts}")
    print(f"  Original qubits: {cr.original_qubits}")

assert "cutting" in result_cutting.passes_applied
assert result_cutting.cutting_result is not None
assert result_cutting.fusion_result is None
assert result_cutting.mitigation_plan is None
print("\nCutting-only assertions passed!")

## 7. Full Pipeline with FakeBrisbane Backend

In [ ]:
backend = FakeBrisbane()
compiler_brisbane = QCompiler(backend=backend)

print(f"Backend: {backend.name}")
print(f"Qubits: {backend.num_qubits}")
print(f"Cost model device: {compiler_brisbane.cost_model.device.backend_name}")
print(f"Cost model qubits: {compiler_brisbane.cost_model.device.num_qubits}")

assert compiler_brisbane.cost_model.device.num_qubits == backend.num_qubits
print("\nBackend initialization assertions passed!")

In [ ]:
circuits_for_backend = {}

bell_hw = QuantumCircuit(2)
bell_hw.h(0)
bell_hw.cx(0, 1)
bell_hw.measure_all()
circuits_for_backend['Bell'] = bell_hw

ghz_hw = QuantumCircuit(4)
ghz_hw.h(0)
for i in range(1, 4):
    ghz_hw.cx(0, i)
ghz_hw.measure_all()
circuits_for_backend['GHZ-4'] = ghz_hw

qft_hw = QFT(4, do_swaps=True).decompose()
qft_hw.measure_all()
circuits_for_backend['QFT-4'] = qft_hw

qaoa_hw = QuantumCircuit(4)
for i in range(4):
    qaoa_hw.h(i)
for i in range(3):
    qaoa_hw.cx(i, i+1)
    qaoa_hw.rz(0.5, i+1)
    qaoa_hw.cx(i, i+1)
for i in range(4):
    qaoa_hw.rx(0.3, i)
qaoa_hw.measure_all()
circuits_for_backend['QAOA-4'] = qaoa_hw

print(f"{'Circuit':<10} {'F Before':>10} {'F After':>10} {'Improvement':>12} {'Passes':>30}")
print("-" * 75)

for name, qc in circuits_for_backend.items():
    r = compiler_brisbane.optimize(qc)
    print(f"{name:<10} {r.fidelity_before:>10.6f} {r.fidelity_after:>10.6f} "
          f"{r.fidelity_improvement:>12.6f} {str(r.passes_applied):>30}")
    assert r.fidelity_before > 0
    assert r.fidelity_after > 0
    assert r.original_circuit is not None
    assert r.optimized_circuit is not None

print("\nAll FakeBrisbane pipeline assertions passed!")

## 8. Custom OptimizerConfig (Selective Passes)

In [ ]:
configs = {
    'Fusion+Scheduling': OptimizerConfig(
        fusion=True, cutting=False, mitigation="none",
        scheduling="coherence_aware", batch=False, autotune=False,
    ),
    'Fusion+Mitigation': OptimizerConfig(
        fusion=True, cutting=False, mitigation="zne",
        scheduling="none", batch=False, autotune=False,
    ),
    'Cutting+Mitigation': OptimizerConfig(
        fusion=False, cutting=True, mitigation="pec",
        scheduling="none", batch=False, autotune=False,
    ),
    'Scheduling+Mitigation': OptimizerConfig(
        fusion=False, cutting=False, mitigation="cdr",
        scheduling="alap", batch=False, autotune=False,
    ),
    'Full minus Batch': OptimizerConfig(
        fusion=True, cutting=True, mitigation="zne",
        scheduling="coherence_aware", batch=False, autotune=False,
    ),
}

qc_custom = QuantumCircuit(3)
qc_custom.h(0)
qc_custom.rz(0.5, 0)
qc_custom.sx(0)
qc_custom.cx(0, 1)
qc_custom.cx(1, 2)

print(f"{'Config':<25} {'Passes':>45}")
print("-" * 72)

for name, cfg in configs.items():
    r = compiler.optimize(qc_custom, config=cfg)
    print(f"{name:<25} {str(r.passes_applied):>45}")
    assert r.config == cfg
    assert r.fidelity_before > 0

print("\nAll custom config assertions passed!")

## 9. Batch Optimization of Multiple Circuits

In [ ]:
batch_circuits = []
for i in range(5):
    qc = QuantumCircuit(2)
    qc.h(0)
    qc.cx(0, 1)
    for _ in range(i):
        qc.rz(0.3 * (i + 1), 0)
        qc.rx(0.2 * (i + 1), 1)
    batch_circuits.append(qc)

batch_config = OptimizerConfig(
    fusion=True,
    cutting=False,
    mitigation="none",
    scheduling="none",
    batch=True,
    autotune=False,
)

batch_results = compiler.optimize_batch(batch_circuits, config=batch_config)

print(f"Batch size: {len(batch_circuits)}")
print(f"Results returned: {len(batch_results)}")
print()
print(f"{'Circuit':>8} {'F Before':>10} {'F After':>10} {'Improvement':>12}")
print("-" * 44)

for i, r in enumerate(batch_results):
    print(f"{i:>8} {r.fidelity_before:>10.6f} {r.fidelity_after:>10.6f} {r.fidelity_improvement:>12.6f}")
    assert r.fidelity_before > 0
    assert r.optimized_circuit is not None

assert all(r.batch_plan is not None for r in batch_results)
print(f"\nBatch plan assigned to all results: {all(r.batch_plan is not None for r in batch_results)}")
print("Batch optimization assertions passed!")

In [ ]:
batch_config_no_batch = OptimizerConfig(
    fusion=True,
    cutting=False,
    mitigation="none",
    scheduling="none",
    batch=False,
    autotune=False,
)

results_no_batch = compiler.optimize_batch(batch_circuits, config=batch_config_no_batch)

assert len(results_no_batch) == len(batch_circuits)
assert all(r.batch_plan is None for r in results_no_batch)
print(f"Without batching: {len(results_no_batch)} results, none have batch_plan")
print("Batch disable assertion passed!")

## 10. Fidelity Improvement Comparison (Before vs After)

In [ ]:
test_circuits = {}

bell_t = QuantumCircuit(2)
bell_t.h(0)
bell_t.cx(0, 1)
bell_t.measure_all()
test_circuits['Bell'] = bell_t

ghz_t = QuantumCircuit(4)
ghz_t.h(0)
for i in range(1, 4):
    ghz_t.cx(0, i)
ghz_t.measure_all()
test_circuits['GHZ-4'] = ghz_t

qft_t = QFT(4, do_swaps=True).decompose()
qft_t.measure_all()
test_circuits['QFT-4'] = qft_t

qaoa_t = QuantumCircuit(4)
for i in range(4):
    qaoa_t.h(i)
for i in range(3):
    qaoa_t.cx(i, i+1)
    qaoa_t.rz(0.5, i+1)
    qaoa_t.cx(i, i+1)
for i in range(4):
    qaoa_t.rx(0.3, i)
qaoa_t.measure_all()
test_circuits['QAOA-4'] = qaoa_t

deep_t = QuantumCircuit(3)
for _ in range(10):
    deep_t.h(0)
    deep_t.cx(0, 1)
    deep_t.cx(1, 2)
deep_t.measure_all()
test_circuits['Deep'] = deep_t

full_config = OptimizerConfig(
    fusion=True, cutting=True, mitigation="zne",
    scheduling="coherence_aware", batch=False, autotune=False,
)

print(f"{'Circuit':<10} {'Qubits':>6} {'F Before':>10} {'F After':>10} {'Delta':>10} {'Pct':>8}")
print("-" * 58)

improvements = []
for name, qc in test_circuits.items():
    r = compiler.optimize(qc, config=full_config)
    delta = r.fidelity_improvement
    pct = r.fidelity_improvement_pct
    improvements.append((name, delta, pct))
    print(f"{name:<10} {qc.num_qubits:>6} {r.fidelity_before:>10.6f} {r.fidelity_after:>10.6f} {delta:>10.6f} {pct:>7.2f}%")
    assert r.fidelity_before > 0
    assert r.fidelity_after > 0

print("\nAll fidelity comparison assertions passed!")

## 11. Edge Cases

In [ ]:
# Edge case: empty circuit
empty = QuantumCircuit(4)
result_empty = compiler.optimize(empty)
print(f"Empty circuit: fidelity_before={result_empty.fidelity_before:.6f}, "
      f"fidelity_after={result_empty.fidelity_after:.6f}")
print(f"  Passes: {result_empty.passes_applied}")
assert result_empty.fidelity_before > 0
assert result_empty.fidelity_after > 0
print("Empty circuit assertion passed!")

In [ ]:
# Edge case: single gate
single = QuantumCircuit(1)
single.h(0)
single.measure_all()
result_single = compiler.optimize(single)
print(f"Single gate: fidelity_before={result_single.fidelity_before:.6f}, "
      f"fidelity_after={result_single.fidelity_after:.6f}")
print(f"  Passes: {result_single.passes_applied}")
assert result_single.fidelity_before > 0
assert result_single.fidelity_after > 0
print("Single gate assertion passed!")

In [ ]:
# Edge case: no passes enabled
no_pass_config = OptimizerConfig(
    fusion=False,
    cutting=False,
    mitigation="none",
    scheduling="none",
    batch=False,
    autotune=False,
)
qc_nopass = QuantumCircuit(2)
qc_nopass.h(0)
qc_nopass.cx(0, 1)

result_nopass = compiler.optimize(qc_nopass, config=no_pass_config)
print(f"No passes: passes_applied={result_nopass.passes_applied}")
print(f"  Fidelity before: {result_nopass.fidelity_before:.6f}")
print(f"  Fidelity after: {result_nopass.fidelity_after:.6f}")
print(f"  Fusion result: {result_nopass.fusion_result}")
print(f"  Cutting result: {result_nopass.cutting_result}")
print(f"  Schedule result: {result_nopass.schedule_result}")
print(f"  Mitigation plan: {result_nopass.mitigation_plan}")

assert len(result_nopass.passes_applied) == 0
assert result_nopass.fusion_result is None
assert result_nopass.cutting_result is None
assert result_nopass.schedule_result is None
assert result_nopass.mitigation_plan is None
assert result_nopass.fidelity_before > 0
assert result_nopass.fidelity_after > 0
print("No-pass assertion passed!")

In [ ]:
# Edge case: circuit with many sequential single-qubit gates (maximizes fusion)
fusion_heavy = QuantumCircuit(2)
fusion_heavy.h(0)
fusion_heavy.rz(0.3, 0)
fusion_heavy.sx(0)
fusion_heavy.rz(0.7, 0)
fusion_heavy.x(0)
fusion_heavy.cx(0, 1)
fusion_heavy.h(1)
fusion_heavy.rz(0.4, 1)
fusion_heavy.sx(1)

result_fh = compiler.optimize(fusion_heavy, config=OptimizerConfig(
    fusion=True, cutting=False, mitigation="none",
    scheduling="none", batch=False, autotune=False,
))
print(f"Fusion-heavy circuit:")
print(f"  Chains fused: {result_fh.fusion_result.chains_fused}")
print(f"  Gates: {result_fh.fusion_result.total_gates_before} -> {result_fh.fusion_result.total_gates_after}")
print(f"  Depth: {result_fh.fusion_result.depth_before} -> {result_fh.fusion_result.depth_after}")
print(f"  Fidelity: {result_fh.fidelity_before:.6f} -> {result_fh.fidelity_after:.6f}")

assert result_fh.fusion_result is not None
print("Fusion-heavy assertion passed!")

## 12. Validation Summary

In [ ]:
print("="" * 60)
print("TRANSPILER PIPELINE VALIDATION SUMMARY")
print("="" * 60)
print()
print("✓ QCompiler and OptimizerConfig imports")
print("✓ Default pipeline (all passes enabled) with Bell and GHZ-4")
print("✓ Fusion-only optimization")
print("✓ Scheduling-only (asap, alap, coherence_aware)")
print("✓ Mitigation-only (zne, pec, cdr, adaptive)")
print("✓ Cutting-only optimization")
print("✓ Full pipeline with FakeBrisbane backend")
print("✓ Custom OptimizerConfig (selective passes)")
print("✓ Batch optimization of multiple circuits")
print("✓ Fidelity improvement comparison (before vs after)")
print("✓ Edge cases (empty circuit, single gate, no passes, fusion-heavy)")
print("✓ QCompilerResult properties (fidelity_improvement, fidelity_improvement_pct)")
print()
total_tests = len(test_circuits) + len(scheduling_methods) + len(mitigation_methods) + len(configs) + len(batch_circuits) + 4
print(f"Total circuits/configurations validated: {total_tests}")
print(f"FakeBrisbane backend: {backend.num_qubits} qubits")